# 03 - Embeddings & Vector Index

Generates dense text embeddings for every catalog book using `intfloat/multilingual-e5-large` and builds a local Chroma vector index under `data/vector_store/`.


In [1]:
import sys
import time
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import chromadb
import pandas as pd
from sentence_transformers import SentenceTransformer

from app.config import (
    CATALOG_CSV,
    EMBEDDING_MODEL_NAME,
    EMBEDDING_PASSAGE_PREFIX,
    EMBEDDING_QUERY_PREFIX,
    VECTOR_STORE_COLLECTION,
    VECTOR_STORE_DIR,
    build_passage_text,
    get_logger,
)

logger = get_logger("embeddings")

catalog = pd.read_csv(CATALOG_CSV, dtype={"isbn13": "string"})
logger.info("Loaded catalog: %d rows, %d columns", *catalog.shape)


2026-08-17 18:29:37 | INFO    | booklens.embeddings | Loaded catalog: 22568 rows, 11 columns


## 1. Construct Passage Text

Builds structured passage text for each book combining title, category, and description.


In [2]:
catalog["_passage_text"] = [
    build_passage_text(row.title, row.categories, row.description)
    for row in catalog.itertuples()
]

empty_text = catalog["_passage_text"] == ""
if empty_text.any():
    logger.warning("%d rows produced an empty passage text - these can't be embedded meaningfully", empty_text.sum())

catalog["_catalog_row_id"] = [str(i) for i in range(len(catalog))]
logger.info("Prepared passage text for %d rows", len(catalog))
catalog[["title", "_passage_text"]].head(3)


2026-08-17 18:29:38 | INFO    | booklens.embeddings | Prepared passage text for 22568 rows


,title,_passage_text
0,Gilead,Gilead . Fiction . A NOVEL THAT READERS and cr...
1,Spider's Web,Spider's Web . Detective and mystery stories ....
2,The One Tree,The One Tree . American fiction . Volume Two o...


## 2. Generate Embeddings

Encodes catalog passages in batches with the `passage: ` prefix.


In [3]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

prefixed_texts = [EMBEDDING_PASSAGE_PREFIX + text for text in catalog["_passage_text"]]

start_time = time.time()
catalog_embeddings = embedding_model.encode(
    prefixed_texts,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
)
elapsed_seconds = time.time() - start_time

logger.info(
    "Embedded %d rows -> shape %s in %.1fs (%.1f rows/sec)",
    len(catalog), catalog_embeddings.shape, elapsed_seconds, len(catalog) / elapsed_seconds,
)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/353 [00:00<?, ?it/s]

2026-08-17 18:41:38 | INFO    | booklens.embeddings | Embedded 22568 rows -> shape (22568, 1024) in 707.4s (31.9 rows/sec)


## 3. Build Chroma Vector Store

Creates a persisted Chroma collection indexed by cosine similarity.


In [4]:
chroma_client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))

try:
    chroma_client.delete_collection(VECTOR_STORE_COLLECTION)
    logger.info("Dropped existing collection %r before rebuilding it", VECTOR_STORE_COLLECTION)
except Exception:
    pass  # collection did not exist yet - nothing to drop

collection = chroma_client.create_collection(
    VECTOR_STORE_COLLECTION,
    metadata={"hnsw:space": "cosine"},
)


2026-08-17 18:41:41 | INFO    | booklens.embeddings | Dropped existing collection 'booklens_catalog' before rebuilding it


In [5]:
BATCH_SIZE = 500

for start in range(0, len(catalog), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(catalog))
    batch = catalog.iloc[start:end]

    collection.add(
        ids=batch["_catalog_row_id"].tolist(),
        embeddings=catalog_embeddings[start:end].tolist(),
        documents=batch["_passage_text"].tolist(),
        metadatas=[
            {
                "title": str(row.title),
                "isbn13": str(row.isbn13) if pd.notna(row.isbn13) else "",
                "source": str(row.source),
            }
            for row in batch.itertuples()
        ],
    )

logger.info("Persisted %d vectors to %s (collection=%r)", collection.count(), VECTOR_STORE_DIR, VECTOR_STORE_COLLECTION)


2026-08-17 18:42:04 | INFO    | booklens.embeddings | Persisted 22568 vectors to /Users/obscure/Developer/MINOR/book_lens/data/vector_store (collection='booklens_catalog')


## 4. Semantic Search Sanity Check

Tests query embedding and similarity retrieval on sample search phrases.


In [6]:
def search(query_text: str, n_results: int = 5):
    query_embedding = embedding_model.encode([EMBEDDING_QUERY_PREFIX + query_text], normalize_embeddings=True)
    results = collection.query(query_embeddings=query_embedding.tolist(), n_results=n_results)
    for row_id, metadata, distance in zip(results["ids"][0], results["metadatas"][0], results["distances"][0]):
        print(f"{1 - distance:.3f}  {metadata['title']}  (row {row_id}, {metadata['source']})")


search("a young wizard discovers he has magical powers")


0.838  The Girl, the Dragon, and the Wild Magic  (row 2613, 7k_books)
0.830  Harry Potter and the Sorcerer's Stone (Harry Potter, Book 1)  (row 12615, google_books)
0.828  Hidden Talents  (row 4608, 7k_books)
0.825  Discoveries  (row 8241, google_books)
0.821  Harry Potter and the Chamber of Secrets by J.K. Rowling  (row 12609, google_books)
